# GitHub Trending Repository Analysis - Exploratory Analysis

This notebook provides an interactive exploration of the GitHub trending repositories dataset and demonstrates various analysis techniques.

## 1. Setup and Data Loading

In [ ]:
import sys
sys.path.insert(0, '../src')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
%matplotlib inline

from src.utils import DataManager
from src.data_processing import process_github_data
from src.analysis import RepositoryAnalyzer
from src.ml_models import ModelPipeline

# Set visualization style
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)

In [ ]:
# Load processed data
data_manager = DataManager()
df = data_manager.load_csv('github_trending_processed.csv')

print(f"Dataset shape: {df.shape}")
print(f"\nColumns: {df.columns.tolist()}")
print(f"\nFirst few rows:")
df.head()

## 2. Data Overview and Summary Statistics

In [ ]:
# Summary statistics
df.describe()

In [ ]:
# Check for missing values
missing_values = df.isnull().sum()
print("Missing Values:")
print(missing_values[missing_values > 0])

## 3. Programming Language Analysis

In [ ]:
# Top languages by repository count
lang_counts = df['language'].value_counts().head(10)
print("Top 10 Languages by Repository Count:")
print(lang_counts)

# Visualize
fig, ax = plt.subplots(figsize=(10, 6))
lang_counts.plot(kind='barh', ax=ax, color='steelblue')
ax.set_xlabel('Number of Repositories')
ax.set_title('Top 10 Programming Languages')
plt.tight_layout()
plt.show()

In [ ]:
# Languages by average stars
lang_stats = df.groupby('language').agg({
    'stars': ['mean', 'median', 'count'],
    'forks': 'mean'
}).round(2)

lang_stats.columns = ['avg_stars', 'median_stars', 'count', 'avg_forks']
lang_stats = lang_stats[lang_stats['count'] >= 5].sort_values('avg_stars', ascending=False)

print("Top Languages by Average Stars:")
print(lang_stats.head(10))

## 4. Correlation Analysis

In [ ]:
# Calculate correlations
numeric_cols = ['stars', 'forks', 'watchers', 'open_issues', 'engagement_score']
corr_matrix = df[numeric_cols].corr()

# Visualize correlation matrix
fig, ax = plt.subplots(figsize=(8, 6))
sns.heatmap(corr_matrix, annot=True, fmt='.2f', cmap='coolwarm', center=0, ax=ax)
ax.set_title('Correlation Matrix of Key Metrics')
plt.tight_layout()
plt.show()

print("\nCorrelation Matrix:")
print(corr_matrix)

## 5. Machine Learning Analysis

In [ ]:
# Run ML pipeline
pipeline = ModelPipeline(df)
ml_results = pipeline.run_full_pipeline()

print("\n✓ ML Pipeline Complete")

In [ ]:
# Cluster profiles
print("Cluster Profiles:")
for cluster_name, profile in ml_results['cluster_profiles'].items():
    print(f"\n{cluster_name}:")
    for key, value in profile.items():
        print(f"  {key}: {value}")

In [ ]:
# Feature importance
print("Feature Importance for Success Prediction:")
feature_imp = ml_results['feature_importance']
for feat, importance in feature_imp.items():
    print(f"  {feat}: {importance:.4f}")

# Visualize
fig, ax = plt.subplots(figsize=(10, 6))
features = list(feature_imp.keys())
importances = list(feature_imp.values())
ax.barh(features, importances, color='coral')
ax.set_xlabel('Importance')
ax.set_title('Feature Importance for Repository Success')
plt.tight_layout()
plt.show()

## 6. Top Repositories

In [ ]:
# Top 15 repositories by stars
top_repos = df.nlargest(15, 'stars')[['full_name', 'description', 'language', 'stars', 'forks', 'watchers']].copy()
top_repos.columns = ['Repository', 'Description', 'Language', 'Stars', 'Forks', 'Watchers']

print("Top 15 Repositories by Stars:")
print(top_repos.to_string(index=False))

## 7. Key Insights and Conclusions

In [ ]:
# Summary insights
print("KEY INSIGHTS")
print("="*50)
print(f"\n1. Dataset Overview:")
print(f"   - Total repositories: {len(df)}")
print(f"   - Unique languages: {df['language'].nunique()}")
print(f"   - Unique licenses: {df['license'].nunique()}")

print(f"\n2. Popularity Metrics:")
print(f"   - Average stars: {df['stars'].mean():.0f}")
print(f"   - Median stars: {df['stars'].median():.0f}")
print(f"   - Most starred: {df.loc[df['stars'].idxmax(), 'full_name']} ({df['stars'].max():.0f} stars)")

print(f"\n3. Language Insights:")
most_common = df['language'].value_counts().index[0]
print(f"   - Most common language: {most_common}")
most_popular = df.groupby('language')['stars'].mean().idxmax()
print(f"   - Most popular language: {most_popular}")

print(f"\n4. ML Insights:")
print(f"   - Success prediction accuracy: {ml_results['predictor_performance']['accuracy']:.3f}")
print(f"   - Top success factor: {list(ml_results['feature_importance'].keys())[0]}")